# pilot analysis

Reads every study log in `data/`, computes the same text-entry metrics the app computes (`src/study/metrics.ts`), summarises by tester and pointing mode, draws one chart, and writes `../public/results.json` for the landing page.

Definitions, identical to the app:

- wpm = (characters typed / 5) / minutes, per phrase
- error rate = levenshtein(target, typed) / len(target)
- corrections = delete letter, delete word, clear selections


In [ ]:
import json, glob, datetime, pathlib
import pandas as pd
import matplotlib.pyplot as plt

DATA = pathlib.Path('data')
OUT = pathlib.Path('../public/results.json')

def levenshtein(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

CORRECTIONS = {'deleteChar', 'deleteWord', 'clear'}

def phrase_metrics(events):
    out, start = [], None
    for i, e in enumerate(events):
        if e['type'] == 'phrase_start':
            start, start_i = e, i
        if e['type'] == 'phrase_end' and start is not None:
            inner = events[start_i + 1:i]
            seconds = (e['t'] - start['t']) / 1000
            typed = e['typed'].strip()
            sel = [x for x in inner if x['type'] == 'select']
            lost, lost_at = 0, None
            for x in inner:
                if x['type'] == 'face_lost' and lost_at is None: lost_at = x['t']
                if x['type'] == 'face_found' and lost_at is not None: lost += x['t'] - lost_at; lost_at = None
            if lost_at is not None: lost += e['t'] - lost_at
            out.append(dict(
                index=e['index'], target=e['target'], typed=typed, seconds=seconds,
                wpm=(len(typed) / 5) / (seconds / 60) if seconds else 0,
                error_rate=levenshtein(e['target'], typed) / len(e['target']),
                selections=len(sel),
                selections_per_char=len(sel) / len(typed) if typed else 0,
                corrections=sum(x['actionKind'] in CORRECTIONS for x in sel),
                face_lost_fraction=(lost / 1000) / seconds if seconds else 0,
            ))
            start = None
    return out


In [ ]:
rows = []
for path in sorted(DATA.glob('*.json')):
    events = json.loads(path.read_text())
    head = next(e for e in events if e['type'] == 'session_start')
    for m in phrase_metrics(events):
        rows.append(dict(file=path.name, tester=head['tester'], mode=head['mode'], confirm=head['confirm'], zones=head['zoneCount'], **m))
df = pd.DataFrame(rows)
print(f'{df.file.nunique()} sessions, {len(df)} phrases')
df[['tester', 'mode', 'target', 'typed', 'seconds', 'wpm', 'error_rate', 'selections', 'corrections']]


In [ ]:
by_session = df.groupby(['tester', 'mode', 'confirm'], as_index=False).agg(
    wpm=('wpm', 'mean'), error_rate=('error_rate', 'mean'),
    selections_per_char=('selections_per_char', 'mean'), face_lost=('face_lost_fraction', 'mean'), phrases=('index', 'count'))
by_mode = df.groupby('mode').agg(wpm=('wpm', 'mean'), error_rate=('error_rate', 'mean'), sessions=('file', 'nunique'))
display(by_session.round(3))
by_mode.round(3)


## words per minute by pointing mode

One measure, so one hue. Bars are the mean over all phrases in that mode, dots are each tester's session mean. No legend, the title names the series.

In [ ]:
INK, MUTED, ACCENT, SURFACE, GRID = '#141311', '#4f4a43', '#b42d1c', '#f4f1ea', '#d9d4c8'
modes = [m for m in ['gaze', 'head'] if m in by_mode.index]
fig, ax = plt.subplots(figsize=(6, 3.6), dpi=150, facecolor=SURFACE)
ax.set_facecolor(SURFACE)
x = range(len(modes))
bars = ax.bar(x, [by_mode.loc[m, 'wpm'] for m in modes], width=0.28, color=ACCENT, zorder=2)
for i, m in enumerate(modes):
    pts = by_session[by_session['mode'] == m]
    ax.scatter([i] * len(pts), pts['wpm'], s=44, color=INK, edgecolor=SURFACE, linewidth=1.5, zorder=3)
    for _, r in pts.iterrows():
        ax.annotate(r['tester'], (i, r['wpm']), xytext=(9, -3), textcoords='offset points', fontsize=8, color=MUTED)
    ax.annotate(f"{by_mode.loc[m, 'wpm']:.1f}", (i, by_mode.loc[m, 'wpm']), xytext=(0, 14), textcoords='offset points', ha='center', fontsize=10, color=INK)
ax.set_xticks(list(x), [m for m in modes])
ax.set_xlim(-0.6, len(modes) - 0.4)
ax.set_ylabel('words per minute', color=MUTED)
ax.set_title('mean words per minute by pointing mode, per-tester session means as dots', loc='left', fontsize=10, color=INK)
ax.yaxis.grid(True, color=GRID, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
for s in ['top', 'right', 'left']: ax.spines[s].set_visible(False)
ax.spines['bottom'].set_color(GRID)
ax.tick_params(colors=MUTED, length=0)
ax.set_ylim(0, max(df['wpm'].max() * 1.25, 1))
fig.tight_layout()
fig.savefig('wpm-by-mode.png', facecolor=SURFACE)
plt.show()


In [ ]:
results = {
    'generatedAt': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'n': int(df['tester'].nunique()),
    'sessions': [
        dict(tester=r.tester, mode=r.mode, confirm=r.confirm, meanWpm=round(r.wpm, 3), meanErrorRate=round(r.error_rate, 4),
             phrases=[{k: (round(v, 4) if isinstance(v, float) else v) for k, v in p.items() if k not in ('file',)}
                      for p in df[(df.tester == r.tester) & (df['mode'] == r.mode) & (df.confirm == r.confirm)]
                          .drop(columns=['tester', 'mode', 'confirm', 'zones']).to_dict('records')])
        for r in by_session.itertuples()
    ],
    'byMode': {m: dict(meanWpm=round(by_mode.loc[m, 'wpm'], 3), meanErrorRate=round(by_mode.loc[m, 'error_rate'], 4), n=int(by_mode.loc[m, 'sessions'])) for m in by_mode.index},
}
OUT.write_text(json.dumps(results, indent=2))
print('wrote', OUT.resolve(), 'n =', results['n'])
